<a href="https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-2-generative-ai/lab-03-pretrain-a-tiny-gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 3 (graded) — Pretrain a tiny GPT
**Course 2: Generative AI and LLMs with Python — Chapter 3: Build a small GPT from scratch**

A nanoGPT-scale decoder-only transformer, trained from nothing on real text, entirely in
this notebook — the whole real loop, at a size that fits a free Colab GPU.

**What you'll submit:** loss curves, a sample gallery across checkpoints, and a short
write-up on what doubling the compute budget would change.

## 1. Data: TinyShakespeare (character-level, with offline fallback)

In [ ]:
import urllib.request

def load_text():
    try:
        url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
        text = urllib.request.urlopen(url, timeout=10).read().decode('utf-8')
        print(f'Loaded TinyShakespeare: {len(text):,} characters.')
        return text
    except Exception as e:
        print(f'Offline fallback engaged ({e}) — a small repeated public-domain-style passage.')
        return ('To be, or not to be, that is the question. ' * 2000)

text = load_text()
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}
vocab_size = len(chars)
print('vocab size:', vocab_size)

encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: ''.join(itos[i] for i in ids)

import torch
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

## 2. The model: a small decoder-only transformer

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

BLOCK_SIZE = 128   # context length
N_EMBED = 128
N_HEAD = 4
N_LAYER = 4
DROPOUT = 0.1

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embed, n_head, block_size, dropout):
        super().__init__()
        self.n_head = n_head
        self.qkv = nn.Linear(n_embed, 3 * n_embed)
        self.proj = nn.Linear(n_embed, n_embed)
        self.drop = nn.Dropout(dropout)
        self.register_buffer('mask', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, S, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        d_k = C // self.n_head
        q = q.view(B, S, self.n_head, d_k).transpose(1, 2)
        k = k.view(B, S, self.n_head, d_k).transpose(1, 2)
        v = v.view(B, S, self.n_head, d_k).transpose(1, 2)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(d_k)
        scores = scores.masked_fill(self.mask[:S, :S] == 0, float('-inf'))
        weights = self.drop(F.softmax(scores, dim=-1))
        out = (weights @ v).transpose(1, 2).contiguous().view(B, S, C)
        return self.proj(out)


class Block(nn.Module):
    """Pre-norm transformer block: attention sub-layer, then MLP sub-layer, each with a
    residual connection — the shape from Chapter 1's on-ramp."""
    def __init__(self, n_embed, n_head, block_size, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embed)
        self.attn = CausalSelfAttention(n_embed, n_head, block_size, dropout)
        self.ln2 = nn.LayerNorm(n_embed)
        self.mlp = nn.Sequential(
            nn.Linear(n_embed, 4 * n_embed), nn.GELU(), nn.Linear(4 * n_embed, n_embed), nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class TinyGPT(nn.Module):
    def __init__(self, vocab_size, n_embed, n_head, n_layer, block_size, dropout):
        super().__init__()
        self.block_size = block_size
        self.tok_embed = nn.Embedding(vocab_size, n_embed)
        self.pos_embed = nn.Embedding(block_size, n_embed)
        self.blocks = nn.Sequential(*[Block(n_embed, n_head, block_size, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embed)
        self.head = nn.Linear(n_embed, vocab_size)

    def forward(self, idx, targets=None):
        B, S = idx.shape
        pos = torch.arange(S, device=idx.device)
        x = self.tok_embed(idx) + self.pos_embed(pos)
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=20):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, [-1]]] = float('-inf')
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, 1)
            idx = torch.cat([idx, next_id], dim=1)
        return idx

model = TinyGPT(vocab_size, N_EMBED, N_HEAD, N_LAYER, BLOCK_SIZE, DROPOUT).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'Model size: {n_params/1e6:.2f}M parameters')

## 3. Data loading + LR schedule (warmup then cosine decay)

In [ ]:
BATCH_SIZE = 32
MAX_STEPS = 2000
WARMUP_STEPS = 100
GRAD_ACCUM_STEPS = 2   # simulates a larger batch than fits in memory at once
EVAL_EVERY = 200

def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - BLOCK_SIZE, (BATCH_SIZE,))
    x = torch.stack([d[i:i + BLOCK_SIZE] for i in ix])
    y = torch.stack([d[i + 1:i + BLOCK_SIZE + 1] for i in ix])
    return x.to(device), y.to(device)

def lr_at_step(step, base_lr=3e-4):
    if step < WARMUP_STEPS:
        return base_lr * step / WARMUP_STEPS
    progress = (step - WARMUP_STEPS) / max(1, MAX_STEPS - WARMUP_STEPS)
    return 0.5 * base_lr * (1 + math.cos(math.pi * progress))

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
scaler = torch.cuda.amp.GradScaler(enabled=(device == 'cuda'))

## 4. Train (checkpointing every eval interval, mixed precision on GPU)

In [ ]:
import mlflow

@torch.no_grad()
def estimate_val_loss(n_batches=20):
    model.eval()
    losses = []
    for _ in range(n_batches):
        xb, yb = get_batch('val')
        _, loss = model(xb, yb)
        losses.append(loss.item())
    model.train()
    return sum(losses) / len(losses)

train_losses, val_losses, checkpoints = [], [], {}

with mlflow.start_run(run_name='tiny-gpt-pretrain'):
    mlflow.log_params({'n_params': n_params, 'block_size': BLOCK_SIZE, 'n_layer': N_LAYER,
                        'n_head': N_HEAD, 'n_embed': N_EMBED, 'max_steps': MAX_STEPS})
    model.train()
    for step in range(MAX_STEPS):
        lr = lr_at_step(step)
        for g in optimizer.param_groups:
            g['lr'] = lr

        optimizer.zero_grad()
        accum_loss = 0.0
        for _ in range(GRAD_ACCUM_STEPS):
            xb, yb = get_batch('train')
            with torch.autocast(device_type='cuda' if device == 'cuda' else 'cpu', enabled=(device == 'cuda')):
                _, loss = model(xb, yb)
                loss = loss / GRAD_ACCUM_STEPS
            scaler.scale(loss).backward()
            accum_loss += loss.item()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        if step % EVAL_EVERY == 0 or step == MAX_STEPS - 1:
            val_loss = estimate_val_loss()
            train_losses.append((step, accum_loss))
            val_losses.append((step, val_loss))
            mlflow.log_metrics({'train_loss': accum_loss, 'val_loss': val_loss, 'lr': lr}, step=step)
            checkpoints[step] = {k: v.clone() for k, v in model.state_dict().items()}
            print(f'step {step:5d}  lr={lr:.2e}  train_loss={accum_loss:.4f}  val_loss={val_loss:.4f}  '
                  f'val_ppl={math.exp(val_loss):.1f}')
    example_idx, _ = get_batch('train')
    mlflow.pytorch.log_model(model, 'model', input_example=example_idx[:1].cpu().numpy(), serialization_format='pickle')

In [ ]:
import matplotlib.pyplot as plt

steps, tl = zip(*train_losses)
_, vl = zip(*val_losses)
plt.plot(steps, tl, label='train'); plt.plot(steps, vl, label='val')
plt.xlabel('step'); plt.ylabel('loss'); plt.legend(); plt.title('Tiny GPT pretraining')
plt.show()

print(f'Final val perplexity: {math.exp(val_losses[-1][1]):.1f}')

## 5. Sample gallery across checkpoints

In [ ]:
context = torch.tensor([encode('ROMEO:')], dtype=torch.long).to(device)

for step in sorted(checkpoints.keys())[::max(1, len(checkpoints) // 4)]:
    model.load_state_dict(checkpoints[step])
    model.eval()
    sample = model.generate(context.clone(), max_new_tokens=120)
    print(f'=== step {step} ===')
    print(decode(sample[0].cpu().tolist()))
    print()

model.load_state_dict(checkpoints[max(checkpoints)])  # end on the fully-trained model

## 6. Write-up (fill in)
What did you actually get for your compute budget — is the model producing recognizable
word-like structure by the end, or still mostly noise? If you doubled `MAX_STEPS` (or the
model size), which would you expect to help more given the current gap between train and val
loss, and why (tie this to the Chinchilla/scaling-law intuition from the chapter)?

_Your answer here._

---
*Beacon AI · AIBits Academy — Chapter 3: Build a small GPT from scratch*